In [ ]:
!jq '. | length' ../data/dataset.json

In [ ]:

!jq ".[$(jq 'keys[]' ../data/corpus_embs/dataset.json | shuf -n 1)]" ../data/corpus_embs/dataset.json

In [ ]:

!jq ".[$(jq 'keys[]' ../data/benchmarks/json/queries.json | shuf -n 1)]" ../data/benchmarks/json/queries.json

In [ ]:
from pathlib import Path
import numpy as np 
import json
import matplotlib.pyplot as plt
import seaborn as sns
from arrowspace import ArrowSpaceBuilder
import time

BASE           = Path("~/code_base/leaf_semantic_engine").expanduser()
dataset_json = BASE / "data/corpus_embs/dataset.json"
nomic_raw_path = BASE / "data/corpus_embs/nomic_embs_raw"
qn_dir         = BASE / "data/benchmarks/query_embs_nomic/raw"
DATASET_PATH   = BASE / "data/corpus_embs/dataset.json"
Q_IDX_PATH     = qn_dir / "queries_index.json"

# ── Fixed config: 768d only, tau=0.75 (established best from prior eval)
CORPUS_PATH = nomic_raw_path / "embeddings_nomic_structured_768d_raw.npy"
Q_EMB_PATH  = qn_dir / "queries_emb_768.npy"

TAU = 0.75
GRAPH_PARAMS = {"eps": 0.625, "k": 20, "topk": 10, "p": 2.0, "sigma": None} # from optimazzation routine in embs_eval.ipynb


In [ ]:
with open(DATASET_PATH) as f:
    dataset = json.load(f)

In [ ]:
corpus_embs = np.load(CORPUS_PATH).astype(np.float64)
query_embs  = np.load(Q_EMB_PATH).astype(np.float64)

In [ ]:
print(f"Corpus embs shape: {corpus_embs.shape}")
print(f"Query embs shape: {query_embs.shape}")

In [ ]:
print(f"document example from dataset: {dataset[0]}\nembedding example from dataset: {corpus_embs[0][:10]}")

# Build arrowspace 

In [ ]:
t0 = time.perf_counter()
aspace, gl = ArrowSpaceBuilder().build(GRAPH_PARAMS, corpus_embs)
t1 = time.perf_counter()
print(f"Time to build arrow space: {t1 - t0:.2f} s")

In [ ]:
for query in query_embs:
    result = aspace.search(query, gl, TAU)
    for idx, score in result:
        print(f"Score: {score:.4f}, Document: {dataset[idx]}")